# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install crewai crewai_tools python-dotenv truststore

- Import libraries, APIs and LLM

In [3]:
from crewai import Agent, Task, Crew, LLM

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [4]:
# Use the Windows certificate store so HTTPS works behind antivirus SSL scanning (e.g. Avast)
import truststore
truststore.inject_into_ssl()

import os
from dotenv import load_dotenv

load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

os.environ["SERPER_API_KEY"] = serper_api_key

# crewAI's native LLM class; the "openrouter/" prefix routes requests to OpenRouter
llm = LLM(
    model="openrouter/openai/gpt-3.5-turbo",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
)

## crewAI Tools

In [5]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [6]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool],
    llm=llm
)

In [7]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool],
    llm=llm
)

In [8]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool],
    llm=llm
)

In [19]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm's risk tolerance.",
    verbose=True,
    allow_delegation=True,
    # tools = [scrape_tool, search_tool],
    llm=llm
)

## Creating Tasks

In [21]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [22]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [23]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [24]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

In [14]:
# langchain_openai is no longer needed: crewAI's LLM class is used for the manager

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [25]:
from crewai import Crew, Process

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=llm,
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [26]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [27]:
### this execution will take some time to run
### this execution will take some time to run
result = await financial_trading_crew.kickoff_async(inputs=financial_trading_inputs)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d749969a-9b2f-433e-b5fa-16f5e7af24b6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling     │
│  and machine learning to identify trends and predict market movements.                                          │
│  ID: 1dcedeec-f0f4-429f-8627-c0f9d9d0fadf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling     │
│  and machine learning to identify trends and predict market movements.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AAPL stock market data analysis'}                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai_tools.tools.serper_dev_tool.serper_dev_tool:Error making request to Serper API: HTTPSConnectionPool(host='google.serper.dev', port=443): Read timed out. (read timeout=10)


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: Based on the information provided, let's evaluate the risks associated with the proposed trading       │
│  strategies for AAPL and suggest mitigation strategies.                                                         │
│                                                                                                                 │
│  1. **Market Risk**: AAPL is subject to market fluctuations, including economic conditions, technological       │
│  advancements, and global events. Given its high market capitalization and revenue, AAPL is vulnerable to       │
│  macroeconomic changes impacting consumer spending or technology trends.                                        │
│                                                                                                                 │
│     *Mitigation Strategy*: Diversification of the portfolio to reduce concentration risk in AAPL stock.         │
│  Utilize options strategies like protective puts or collars to hedge against downside risk.                     │
│                                                                                                                 │
│  2. **Stock-Specific Risk**: AAPL's performance can be influenced by company-specific factors such as product   │
│  launches, supply chain disruptions, or legal challenges. Any negative news related to Apple's products or      │
│  leadership could impact the stock price.                                                                       │
│                                                                                                                 │
│     *Mitigation Strategy*: Stay updated on company news and earnings reports. Consider setting stop-loss        │
│  orders to limit potential losses in case of adverse developments.                                              │
│                                                                                                                 │
│  3. **Liquidity Risk**: AAPL's trading volume may fluctuate, affecting the ease of buying or selling shares     │
│  without significantly impacting the stock price. Illiquidity can lead to wider bid-ask spreads and difficulty  │
│  in executing trades at desired prices.                                                                         │
│                                                                                                                 │
│     *Mitigation Strategy*: Monitor trading volumes and average daily trading ranges. Consider using limit       │
│  orders to control the execution price and avoid slippage in illiquid market conditions.                        │
│                                                                                                                 │
│  4. **Regulatory Risk**: Changes in regulations related to technology, trade policies, or intellectual          │
│  property rights can affect AAPL's business operations and financial performance. Legal disputes or regulatory  │
│  investigations could create uncertainty for investors.                                                         │
│                                                                                                                 │
│     *Mitigation Strategy*: Stay informed about regulatory developments and their potential impact on AAPL.      │
│  Evaluate the company's compliance measures and legal standing to assess regulatory risks accurately.           │
│                                                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The risks associated with the proposed trading strategies for AAPL have been evaluated, and mitigation         │
│  strategies have been suggested. Here is the comprehensive risk analysis report:                                │
│                                                                                                                 │
│  1. **Market Risk**:                                                                                            │
│     - **Analysis**: AAPL is exposed to market fluctuations due to economic conditions and technological         │
│  advancements.                                                                                                  │
│     - **Mitigation**: Diversify the portfolio and utilize options strategies for hedging.                       │
│                                                                                                                 │
│  2. **Stock-Specific Risk**:                                                                                    │
│     - **Analysis**: AAPL's performance can be influenced by product launches and legal challenges.              │
│     - **Mitigation**: Stay updated on company news and consider setting stop-loss orders.                       │
│                                                                                                                 │
│  3. **Liquidity Risk**:                                                                                         │
│     - **Analysis**: Trading volume fluctuations can impact ease of buying/selling shares.                       │
│     - **Mitigation**: Monitor trading volumes and use limit orders in illiquid market conditions.               │
│                                                                                                                 │
│  4. **Regulatory Risk**:                                                                                        │
│     - **Analysis**: Changes in regulations can affect AAPL's operations and financial performance.              │
│     - **Mitigation**: Stay informed about regulatory developments and evaluate compliance measures.             │
│                                                                                                                 │
│  5. **Currency Risk**:                                                                                          │
│     - **Analysis**: Currency fluctuations can impact AAPL's revenue and earnings.                               │
│     - **Mitigation**: Hedge currency risk using financial instruments and monitor foreign exchange markets.     │
│                                                                                                                 │
│  In summary, while AAPL presents a promising investment opportunity, it's crucial to manage market,             │
│  stock-specific, liquidity, regulatory, and currency risks effectively. By implementing the suggested           │
│  mitigation strategies and staying informed about market dynamics, investors can make informed trading          │
│  decisions and navigate potential risks successfully.                                                           │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate the risks associated with the proposed trading strategies and execution plans for AAPL.         │
│  Provide a detailed analysis of potential risks and suggest mitigation strategies.                              │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d749969a-9b2f-433e-b5fa-16f5e7af24b6                                                                       │
│  Final Output: The risks associated with the proposed trading strategies for AAPL have been evaluated, and      │
│  mitigation strategies have been suggested. Here is the comprehensive risk analysis report:                     │
│                                                                                                                 │
│  1. **Market Risk**:                                                                                            │
│     - **Analysis**: AAPL is exposed to market fluctuations due to economic conditions and technological         │
│  advancements.                                                                                                  │
│     - **Mitigation**: Diversify the portfolio and utilize options strategies for hedging.                       │
│                                                                                                                 │
│  2. **Stock-Specific Risk**:                                                                                    │
│     - **Analysis**: AAPL's performance can be influenced by product launches and legal challenges.              │
│     - **Mitigation**: Stay updated on company news and consider setting stop-loss orders.                       │
│                                                                                                                 │
│  3. **Liquidity Risk**:                                                                                         │
│     - **Analysis**: Trading volume fluctuations can impact ease of buying/selling shares.                       │
│     - **Mitigation**: Monitor trading volumes and use limit orders in illiquid market conditions.               │
│                                                                                                                 │
│  4. **Regulatory Risk**:                                                                                        │
│     - **Analysis**: Changes in regulations can affect AAPL's operations and financial performance.              │
│     - **Mitigation**: Stay informed about regulatory developments and evaluate compliance measures.             │
│                                                                                                                 │
│  5. **Currency Risk**:                                                                                          │
│     - **Analysis**: Currency fluctuations can impact AAPL's revenue and earnings.                               │
│     - **Mitigation**: Hedge currency risk using financial instruments and monitor foreign exchange markets.     │
│                                                                                                                 │
│  In summary, while AAPL presents a promising investment opportunity, it's crucial to manage market,             │
│  stock-specific, liquidity, regulatory, and currency risks effectively. By implementing the suggested           │
│  mitigation strategies and staying informed about market dynamics, investors can make informed trading          │
│  decisions and navigate potential risks successfully.                                                           │
│                                                                                                                 │
│                                                       

- Display the final result as Markdown.

In [28]:
from IPython.display import Markdown
Markdown(result.raw)

The risks associated with the proposed trading strategies for AAPL have been evaluated, and mitigation strategies have been suggested. Here is the comprehensive risk analysis report:

1. **Market Risk**:
   - **Analysis**: AAPL is exposed to market fluctuations due to economic conditions and technological advancements.
   - **Mitigation**: Diversify the portfolio and utilize options strategies for hedging.

2. **Stock-Specific Risk**:
   - **Analysis**: AAPL's performance can be influenced by product launches and legal challenges.
   - **Mitigation**: Stay updated on company news and consider setting stop-loss orders.

3. **Liquidity Risk**:
   - **Analysis**: Trading volume fluctuations can impact ease of buying/selling shares.
   - **Mitigation**: Monitor trading volumes and use limit orders in illiquid market conditions.

4. **Regulatory Risk**:
   - **Analysis**: Changes in regulations can affect AAPL's operations and financial performance.
   - **Mitigation**: Stay informed about regulatory developments and evaluate compliance measures.

5. **Currency Risk**:
   - **Analysis**: Currency fluctuations can impact AAPL's revenue and earnings.
   - **Mitigation**: Hedge currency risk using financial instruments and monitor foreign exchange markets.

In summary, while AAPL presents a promising investment opportunity, it's crucial to manage market, stock-specific, liquidity, regulatory, and currency risks effectively. By implementing the suggested mitigation strategies and staying informed about market dynamics, investors can make informed trading decisions and navigate potential risks successfully.